In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import seaborn as sns
import h5py
import time
import random
import gc
import shap
import torch
from IPython.display import clear_output
import re
#import anndata as ad
#!pip install pyaging
#import pyaging as pya
!pip uninstall -y seaborn
!pip install seaborn --upgrade
clear_output()
!pip install pacmap
clear_output()
!pip install adjustText
clear_output()
import pacmap
from adjustText import adjust_text
gc.collect()
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import LinearSVR, NuSVR, SVR
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score
from scipy.stats import t as student_t
from catboost import CatBoostRegressor,CatBoostClassifier
from lightgbm import LGBMRegressor
!pip install pytorch_tabnet
clear_output()
from pytorch_tabnet.tab_model import TabNetRegressor

## Loading Data

In [ ]:
def load_idmap(idmap_dir):
    idmap = pd.read_csv(idmap_dir, sep=",")
    age = idmap.age.to_numpy()
    age = age.astype(np.float32)
    sample_type = idmap.disease.replace({"control":0, "Alzheimer's disease":1, "schizophrenia":2, "Parkinson's disease":3, "rheumatoid arthritis":4,
                                         "stroke":5, "Huntington's disease":6, "Graves' disease":7, "type 2 diabetes":8, "Sjogren's syndrome":9})
    return age, sample_type

y, sample_type = load_idmap("/kaggle/input/age-assessment-and-disease-risk-prediction-h5/trainmap.csv")
all_ind = pd.DataFrame(sample_type)

In [ ]:
control_ind = list(all_ind[all_ind['disease']==0].index)
alzhei_ind = list(all_ind[all_ind['disease']==1].index)
schizo_ind = list(all_ind[all_ind['disease']==2].index)
parkin_ind = list(all_ind[all_ind['disease']==3].index)
rheuma_ind = list(all_ind[all_ind['disease']==4].index)
stroke_ind = list(all_ind[all_ind['disease']==5].index)
huntin_ind = list(all_ind[all_ind['disease']==6].index)
graves_ind = list(all_ind[all_ind['disease']==7].index)
diabet_ind = list(all_ind[all_ind['disease']==8].index)
sjogre_ind = list(all_ind[all_ind['disease']==9].index)
control_y = y[control_ind]
alzhei_y = y[alzhei_ind]
schizo_y = y[schizo_ind]
parkin_y = y[parkin_ind]
rheuma_y = y[rheuma_ind]
stroke_y = y[stroke_ind]
huntin_y = y[huntin_ind]
graves_y = y[graves_ind]
diabet_y = y[diabet_ind]
sjogre_y = y[sjogre_ind]

In [ ]:
idmap_path = "/kaggle/input/age-assessment-and-disease-risk-prediction-h5/trainmap.csv"
idmap = pd.read_csv(idmap_path)
cohort_names = { "control": "Control", "Alzheimer's disease": "Alzheimer's", "schizophrenia": "Schizophrenia", "Parkinson's disease": "Parkinson's", "rheumatoid arthritis": "Rheumatoid Arthritis", "stroke": "Stroke", "Huntington's disease": "Huntington's", "Graves' disease": "Graves'", "type 2 diabetes": "Type 2 Diabetes", "Sjogren's syndrome": "Sjögren's" }
idmap["cohort"] = idmap["disease"].replace(cohort_names)
idmap["sex"] = ( idmap["gender"] .astype(str) .str.strip() .str.upper() .replace({ "FEMALE": "F", "MALE": "M", "NAN": np.nan, "NONE": np.nan, "UNKNOWN": np.nan }) )
cohort_order = [ "Control", "Alzheimer's", "Schizophrenia", "Parkinson's", "Rheumatoid Arthritis", "Stroke", "Huntington's", "Graves'", "Type 2 Diabetes", "Sjögren's" ]
rows = []
for cohort in cohort_order:
    g = idmap[idmap["cohort"] == cohort].copy()
    n = len(g)
    if n == 0:
        continue
    age = g["age"].dropna()
    female_n = (g["sex"] == "F").sum()
    male_n = (g["sex"] == "M").sum()
    unknown_n = n - female_n - male_n
    sample_types = ( g["sample_type"] .dropna() .astype(str) .sort_values() .unique() )
    rows.append({ "Cohort": cohort, "N": n, "Age mean": age.mean(), "Age SD": age.std(), "Age median": age.median(), "Age Q1": age.quantile(0.25), "Age Q3": age.quantile(0.75), "Age min": age.min(), "Age max": age.max(), "Female N": female_n, "Female %": 100 * female_n / n, "Male N": male_n, "Male %": 100 * male_n / n, "Unknown sex N": unknown_n, "Sample type": "; ".join(sample_types), "Ancestry": "Not available" })
cohort_table = pd.DataFrame(rows)
round_cols = [ "Age mean", "Age SD", "Age median", "Age Q1", "Age Q3", "Age min", "Age max", "Female %", "Male %" ]
cohort_table[round_cols] = cohort_table[round_cols].round(2)
cohort_table.to_csv("/kaggle/working/cohort_demographics.csv",index=False)
cohort_table

In [ ]:
idmap_path = "/kaggle/input/age-assessment-and-disease-risk-prediction-h5/trainmap.csv"
idmap = pd.read_csv(idmap_path)
idmap["cohort"] = idmap["disease"].replace({"control": "Control", "Alzheimer's disease": "Alzheimer's", "schizophrenia": "Schizophrenia", "Parkinson's disease": "Parkinson's", "rheumatoid arthritis": "Rheumatoid Arthritis", "stroke": "Stroke", "Huntington's disease": "Huntington's", "Graves' disease": "Graves'", "type 2 diabetes": "Type 2 Diabetes", "Sjogren's syndrome": "Sjögren's"})
idmap["gender_clean"] = (idmap["gender"].astype(str).str.strip().str.upper().replace({"FEMALE": "F", "MALE": "M"}))

def summarize_cohort(g):
    age = g["age"].dropna()
    n = len(g)
    female_n = (g["gender_clean"] == "F").sum()
    male_n = (g["gender_clean"] == "M").sum()
    unknown_n = n - female_n - male_n
    return pd.Series({
        "N": n,
        "Age mean": age.mean(),
        "Age SD": age.std(),
        "Age median": age.median(),
        "Age Q1": age.quantile(0.25),
        "Age Q3": age.quantile(0.75),
        "Age min": age.min(),
        "Age max": age.max(),
        "Female N": female_n,
        "Female %": 100 * female_n / n if n else np.nan,
        "Male N": male_n,
        "Male %": 100 * male_n / n if n else np.nan,
        "Unknown sex N": unknown_n,
        "Sample type(s)": "; ".join(sorted(g["sample_type"].dropna().astype(str).unique())),
        "Ancestry": "Not available in released metadata"
    })

cohort_table = (idmap.groupby("cohort", dropna=False).apply(summarize_cohort).reset_index())
for col in ["Age mean", "Age SD", "Age median", "Age Q1", "Age Q3", "Female %", "Male %"]:
    cohort_table[col] = cohort_table[col].round(2)
cohort_table.to_csv("/kaggle/working/supplementary_cohort_demographics.csv", index=False)
cohort_table

In [ ]:
full_data = h5py.File("/kaggle/input/age-assessment-and-disease-risk-prediction-h5/train.h5", "r")["data"]
X = full_data[control_ind]
inds = list(range(X.shape[0]))
h5py.File("/kaggle/input/age-assessment-and-disease-risk-prediction-h5/train.h5", "r").close()
del full_data
gc.collect()
clear_output()

In [ ]:
rand_train = random.sample(inds, 4000)
X_train, y_train = X[rand_train], control_y[rand_train]
remaining_inds = list(set(inds) - set(rand_train))
rand_test = random.sample(remaining_inds, 2000)
X_test, y_test = X[rand_test], control_y[rand_test]
gc.collect()
clear_output()

In [ ]:
cpg_lookup = pd.read_csv("/kaggle/input/holistic-age-prediction-using-dna-methylation-data/Processed CpG Information.csv")
cpg_lookup = cpg_lookup.dropna(how='all')
mp = cpg_lookup.isnull().mean() * 100
mp = mp[mp > 0].sort_values(ascending=False)
print(mp.round(2))

In [ ]:
with open('/kaggle/input/holistic-age-prediction-using-dna-methylation-data/CpG Site Lookup.txt', 'r') as file:
    cpg_storage = [line.strip() for line in file]
cpg_storage = np.array(cpg_storage)

def get_cpg_info(cpg_site):
    return cpg_lookup[cpg_lookup['Name']==cpg_site].dropna(axis=1)

## Feature Selection

In [ ]:
del X
gc.collect()
clear_output()
lr = Ridge(solver="saga",alpha=100,random_state=42,tol=0.01)
lr.fit(X_train,y_train)
print(mean_absolute_error(lr.predict(X_test),y_test))

In [ ]:
plt.figure(figsize=(8,5), dpi=96)
plt.yscale("log")
sns.histplot(lr.coef_, bins=60, color="#40B0FF")
plt.xlabel("Coefficient Value")
plt.ylabel("Logarithm of CpG Count")
plt.title("Distribution of Ridge Regression Coefficients")
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
coefs = list(lr.coef_)
coefs = [abs(coef) for coef in coefs]
cutoff = round(np.percentile(coefs, 99),5)
print(f"Top 1% cutoff is ±{cutoff}.")
indices = []
for coef,ind in zip(lr.coef_,range(len(lr.coef_))):
    if abs(coef)>cutoff:
        indices.append(ind)
cpg_data = cpg_storage[indices]
X_train = X_train[:,indices]
X_test = X_test[:,indices]

In [ ]:
#file_path = "/kaggle/working/output.txt"
#with open(file_path, 'w') as file:
#    file.write('\n'.join(cpg_data))

In [ ]:
full_data = h5py.File("/kaggle/input/age-assessment-and-disease-risk-prediction-h5/train.h5", "r")["data"]
X = full_data[control_ind]
X_health = full_data[control_ind][:,indices]
X_alzhei = full_data[alzhei_ind][:,indices]
X_schizo = full_data[schizo_ind][:,indices]
X_parkin = full_data[parkin_ind][:,indices]
X_rheuma = full_data[rheuma_ind][:,indices]
X_stroke = full_data[stroke_ind][:,indices]
X_huntin = full_data[huntin_ind][:,indices]
X_graves = full_data[graves_ind][:,indices]
X_diabet = full_data[diabet_ind][:,indices]
X_sjogre = full_data[sjogre_ind][:,indices]
del full_data
gc.collect()
clear_output()

In [ ]:
X_visual = np.concatenate([X_health,X_alzhei,X_schizo,X_parkin,X_rheuma,X_stroke,X_huntin,X_graves,X_diabet,X_sjogre], axis=0)
y_visual = np.concatenate([np.full(X_health.shape[0], 0), np.full(X_alzhei.shape[0], 1), np.full(X_schizo.shape[0], 2), np.full(X_parkin.shape[0], 3), np.full(X_rheuma.shape[0], 4), np.full(X_stroke.shape[0], 5), np.full(X_huntin.shape[0], 6), np.full(X_graves.shape[0], 7), np.full(X_diabet.shape[0], 8), np.full(X_sjogre.shape[0], 9)],axis=0).astype(int)
hex_colors = ["#d5d5d5", "#ffa000", "#0000ff", "#00c000", "#ff0000", "#00c0ff", "#ff00ff", "#005f73", "#8000ff", "#800000"]
cmap_disease = ListedColormap(hex_colors, name="disease10")
conditions = disease_names = ["Control", "Alzheimer's", "Schizophrenia", "Parkinson's", "Rheumatoid Arthritis", "Stroke", "Huntington's", "Graves'", "Type 2 Diabetes", "Sjögren's"]
x,y = np.random.randn(10),np.random.randn(10)

In [ ]:
embedding = pacmap.PaCMAP(n_components=2, n_neighbors=10, MN_ratio=0.5, FP_ratio=2.0)
X_trans = embedding.fit_transform(X_visual, init="pca")
fig, ax = plt.subplots(1, 1, figsize=(5.54, 3.4), dpi=100)
sc = ax.scatter(X_trans[:, 0], X_trans[:, 1], c=y_visual, cmap=cmap_disease, s=3, alpha=0.6)
ax.set_xticks([])
ax.set_xticklabels([])
ax.set_yticks([])
ax.set_yticklabels([])
handles = [Line2D([0],[0], marker="o", linestyle="", color=hex_colors[i], markersize=6, label=disease_names[i]) for i in range(10)]
ax.legend(handles=handles, title="Diagnosis", bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0)
plt.tight_layout()
plt.show()

In [ ]:
embedding = pacmap.PaCMAP(n_components=2, n_neighbors=10, MN_ratio=0.5, FP_ratio=2.0)
X_trans = embedding.fit_transform(X_health, init="pca")
fig, ax = plt.subplots(1, 1, figsize=(4.37, 3.5), dpi=100)
sc = ax.scatter(X_trans[:, 0], X_trans[:, 1], cmap='turbo', c=control_y, s=3, alpha=0.6) # rainbow
ax.set_xticks([])
ax.set_xticklabels([])
ax.set_yticks([])
ax.set_yticklabels([])
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Age (Years)")
plt.tight_layout()
plt.show()

In [ ]:
del X_trans,X
gc.collect()
clear_output()

## Retraining Top 4 Models

In [ ]:
cat_model = CatBoostRegressor(iterations=3200,min_data_in_leaf=1,learning_rate=0.03,max_leaves=50,l2_leaf_reg=0.1,grow_policy='Lossguide',logging_level='Silent',task_type='GPU')
cat_model.fit(X_train, y_train)
cat_preds = cat_model.predict(X_test)
cat_mae = mean_absolute_error(cat_preds,y_test)
print(round(cat_mae,3))

In [ ]:
lgbm_model = LGBMRegressor(n_estimators=150, learning_rate=0.1, num_leaves=80, subsample=0.9)
lgbm_model.fit(X_train, y_train)
lgbm_preds = lgbm_model.predict(X_test)
lgbm_mae = mean_absolute_error(lgbm_preds,y_test)
print(round(lgbm_mae,3))

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=12, learning_rate=0.6, max_depth=17, init=SVR(kernel='poly',degree=2,shrinking=True,gamma='auto',C=1.5,epsilon=0.1,tol=0.01,coef0=0.24,max_iter=2500))
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_mae = mean_absolute_error(gb_preds,y_test)
print(round(gb_mae,3))

In [ ]:
tab_model = TabNetRegressor(momentum=0.02,gamma=1.2,lambda_sparse=0.00015,n_d=12,n_a=8,device_name='cuda')
tab_model.fit(X_train, y_train.reshape(-1, 1), eval_metric=['mae'], eval_set=[(X_test,y_test.reshape(-1, 1))],max_epochs=200,patience=20)
clear_output()
tab_preds = tab_model.predict(X_test)[:,0]
tab_mae = mean_absolute_error(tab_preds,y_test)
print(round(tab_mae,3))

In [ ]:
"""clocks = ["Horvath2013","SkinAndBlood","AltumAge"]

feature_names = None
feature_source = None

for name in ("cpg_storage", "cpg_data"):
    if name in globals():
        arr = np.asarray(globals()[name]).astype(str)
        if len(arr) == X_test.shape[1]:
            feature_names = arr
            feature_source = name
            break

X_test_arr = np.asarray(X_test)
adata = ad.AnnData(X=X_test_arr)
adata.var_names = pd.Index(feature_names, dtype=str)
adata.obs_names = pd.Index([f"sample_{i}" for i in range(adata.n_obs)], dtype=str)
adata = pya.pred.predict_age(adata,clock_names=clocks,batch_size=128,clean=True,verbose=True)"""

In [ ]:
"""def _norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

obs_lookup = {_norm(col): col for col in adata.obs.columns}

rows = []
for clock in clocks:
    pred_col = obs_lookup.get(_norm(clock))
    if pred_col is None:
        raise KeyError(f"{clock} failed. Available columns are {list(adata.obs.columns)}")
    y_pred = adata.obs[pred_col].to_numpy(dtype=float)
    mae = mean_absolute_error(y_test,y_pred)
    rows.append({"Clock": clock, "PredictionColumn": pred_col, "MAE": mae, "PercentNA": adata.uns.get(f"{pred_col}_percent_na", np.nan), "MissingFeatures": len(adata.uns.get(f"{pred_col}_missing_features", []))})

results = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
print("\nMAE results:")
print(results[["Clock", "MAE", "PercentNA", "NumMissingFeatures"]].to_string(index=False))"""

## Error Visualizations

In [ ]:
def three_error_graphs(model_preds,real_values,model_name="",disease_name=""):
    if disease_name=="":
        titles = [f"Actual vs Predicted Age with {model_name}",f"{model_name}'s Residual Error vs Age",f"{model_name}'s Mean Absolute Error by Age Bin"]
    else:
        titles = [f"Actual vs Predicted Age for {disease_name}", f"Residual Error vs Age for {disease_name}", f"Mean Absolute Error by Age Bin for {disease_name}"]
        mae = round(mean_absolute_error(model_preds,real_values),3)
        results[disease_name] = mae
        print("Mean Ensemble had MAE of", mae, "for", disease_name)
    plt.figure(figsize=(6,6),dpi=96)
    sns.scatterplot(x=real_values, y=model_preds, alpha=0.6, color='#0080ff')
    plt.plot([min(real_values), max(real_values)], [min(real_values), max(real_values)], linestyle='--', color='#590099', linewidth=2.5)
    plt.xlabel("Chronological Age")
    plt.ylabel("Predicted Age")
    plt.title(titles[0])
    plt.show()
    errors = real_values-model_preds
    if disease_name!="":
        residuals[disease_name] = errors
    plt.figure(figsize=(7,5), dpi=96)
    sns.scatterplot(x=real_values, y=errors, color='#0080ff', alpha=0.6)
    plt.axhline(0, linestyle='--', color='#590099', linewidth=2.5)
    plt.xlabel("Chronological Age")
    plt.ylabel("Residual Error (Actual - Predicted)")
    plt.title(titles[1])
    plt.show()
    lbound,hbound = max(0,min(real_values)),min(100,max(real_values))
    step = (hbound-lbound)//5
    if disease_name=="":
        bins = pd.cut(real_values, bins=[0, 20, 40, 60, 80, 100])
    elif disease_name=='Type 2 Diabetes':
        bins = pd.cut(real_values, bins=[65, 70, 75, 80, 85, 90])
    else:
        bins = pd.cut(real_values, bins=list(map(int,list(np.arange(lbound, hbound, step)))))
    bin_mae = pd.Series(index=bins.categories, dtype=float)
    for b in bins.categories:
        mask = bins == b
        if mask.sum() == 0:
            bin_mae[b] = 0.0
        else:
            bin_mae[b] = mean_absolute_error(real_values[mask], model_preds[mask])
    plt.figure(figsize=(6, 4.5), dpi=96)
    bin_mae.plot(
        kind='bar',
        color='#40B0FF',
        edgecolor='black'
    )
    plt.title(titles[2])
    plt.ylabel("MAE (years)")
    plt.xlabel("Age Bin")
    plt.xticks(rotation=0, ha='center')
    plt.show()

In [ ]:
three_error_graphs(cat_preds,y_test,model_name='CatBoost')

In [ ]:
three_error_graphs(lgbm_preds,y_test,model_name='LightGBM')

In [ ]:
three_error_graphs(gb_preds,y_test,model_name='Gradient Boosting')

In [ ]:
three_error_graphs(tab_preds,y_test,model_name='TabNet')

## Interpretability and CpG Site Selection

In [ ]:
explainer = shap.Explainer(cat_model)
control_shap = explainer(X_test)
control_shap.feature_names = cpg_data
shap.summary_plot(control_shap, X_test, max_display=15)

In [ ]:
explainer = shap.Explainer(lgbm_model)
control_shap = explainer(X_test)
control_shap.feature_names = cpg_data
shap.summary_plot(control_shap, X_test, max_display=15)

In [ ]:
importances = gb_model.feature_importances_
feature_names = cpg_data
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(7, 5))
plt.barh(fi_df['Feature'], fi_df['Importance'], color='#40B0FF')
plt.xlabel("Feature Importance")
plt.title("Gradient Boosting Top 15 Feature Importances")
plt.gca().invert_yaxis()
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
X_test_tensor = torch.tensor(X_test.astype(np.float32)).to(tab_model.device_name)
M_explain, masks = tab_model.explain(X_test_tensor, normalize=False)
global_importance = np.mean(M_explain, axis=0)
top_indices = np.argsort(global_importance)[::-1][:15]
top_features = [cpg_data[i] for i in top_indices]
top_values = global_importance[top_indices]

pd.Series(top_values, index=top_features).plot(
    kind='barh',
    color='#40B0FF',
    figsize=(7, 5)
)
plt.title("TabNet Top 15 Feature Importances")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
get_cpg_info('cg24673101')

## Mean Ensembling

In [ ]:
df_preds = pd.DataFrame({
    "CatBoost": cat_preds,
    "LGBM": lgbm_preds,
    "GB": gb_preds,
    "TabNet": tab_preds
})
corr_matrix = df_preds.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap='coolwarm', square=True, vmin=0.95, vmax=1.0)
plt.title("Correlation of Models' Predictions")
plt.tight_layout()
plt.show()

In [ ]:
inv_maes = 1 / np.array([cat_mae, lgbm_mae, gb_mae, tab_mae])**2
weights = inv_maes / inv_maes.sum()
print("Model Weights are", [round(weight,3) for weight in weights])

In [ ]:
def ensemble_preds(X_disease):
    cat_dpreds = cat_model.predict(X_disease)
    lgbm_dpreds = lgbm_model.predict(X_disease)
    gb_dpreds = gb_model.predict(X_disease)
    tab_dpreds = tab_model.predict(X_disease)[:,0]
    return (weights[0]*cat_dpreds + weights[1]*lgbm_dpreds + weights[2]*gb_dpreds + weights[3]*tab_dpreds)

## Clinical Trial Candidates

In [ ]:
candidate_gene_map = {"cg20002504": "ANKMY1", "cg09748749": "ASL", "cg04329870": "CD8A", "cg12373771": "CECR6", "cg24369989": "CHRNB4", "cg22156456": "EIF1", "cg11265839": "ELK3", "cg23606718": "AMER3", "cg22454769": "FHL2", "cg09761247": "IDS", "cg01820374": "LAG3", "cg23479922": "MARCH11", "cg07194250": "MIR503", "cg22736354": "NHLRC1", "cg08369368": "NSD1", "cg04875128": "OTUD7A", "cg23813012": "PRDM2", "cg18501647": "PRRT1", "cg06493994": "SCGN", "cg16069986": "SHANK2", "cg17243289": "SMAD2", "cg07553761": "TRIM59", "cg26242531": "ZFYVE21", "cg03664992": "BMP8A", "cg13806070": "BMP8A", "cg16867657": "ELOVL2", "cg21572722": "ELOVL2", "cg24724428": "ELOVL2", "cg08097417": "KLF14", "cg14361627": "KLF14"}

In [ ]:
def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked_p = p_values[order]
    q = ranked_p * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    q_values = np.empty(n, dtype=float)
    q_values[order] = q
    return q_values

In [ ]:
def get_primary_gene(cg):
    gene = ""
    try:
        raw = get_cpg_info(cg)["UCSC_RefGene_Name"].iloc[0]
        raw = "" if raw is None else str(raw)
        if raw and raw.lower() != "nan":
            gene = raw.split(";")[0].strip()
        if gene=="C22orf26" or gene=="LOC150381":
            return "PRR34"
        if gene=="C11orf21":
            return "TSPAN32"
    except:
        gene = ""
    return gene

In [ ]:
def get_annotated_extremes(df, n_each=7):
    neg_rows = []
    pos_rows = []
    for _, row in df.sort_values("EffectSize", ascending=True).iterrows():
        gene = get_primary_gene(row["CpG"])
        if gene != "":
            row_copy = row.copy()
            row_copy["AnnotationGene"] = gene
            neg_rows.append(row_copy)
            if len(neg_rows) == n_each:
                break
    for _, row in df.sort_values("EffectSize", ascending=False).iterrows():
        gene = get_primary_gene(row["CpG"])
        if gene != "":
            row_copy = row.copy()
            row_copy["AnnotationGene"] = gene
            pos_rows.append(row_copy)
            if len(pos_rows) == n_each:
                break
    label_df = pd.DataFrame(neg_rows + pos_rows).drop_duplicates(subset="CpG")
    return label_df

In [ ]:
X_sel = np.asarray(X_health, dtype=np.float64)
age = np.asarray(control_y, dtype=np.float64)
y_centered = age - age.mean()
X_centered = X_sel - X_sel.mean(axis=0, keepdims=True)
ss_y = np.sum(y_centered ** 2)
ss_x = np.sum(X_centered ** 2, axis=0)
slope_per_year = (y_centered @ X_centered) / ss_y
r = (y_centered @ X_centered) / np.sqrt(ss_y * ss_x)
r = np.nan_to_num(r, nan=0.0)
r = np.clip(r, -0.999999999, 0.999999999)
n = len(age)
t_stat = r * np.sqrt((n - 2) / np.maximum(1 - r**2, 1e-300))
p_values = 2 * student_t.sf(np.abs(t_stat), df=n - 2)
q_values = benjamini_hochberg(p_values)
ewas_df = pd.DataFrame({"CpG": cpg_data, "EffectSize": slope_per_year, "Correlation": r, "PValue": p_values, "FDR": q_values})
ewas_df["neglog10P"] = -np.log10(np.clip(ewas_df["PValue"], 1e-300, 1.0))
ewas_df["Status"] = "Other CpGs"
ewas_df.loc[(ewas_df["FDR"] < 0.05) & (ewas_df["EffectSize"] > 0), "Status"] = "Age-associated hypermethylation"
ewas_df.loc[(ewas_df["FDR"] < 0.05) & (ewas_df["EffectSize"] < 0), "Status"] = "Age-associated hypomethylation"
label_df = get_annotated_extremes(ewas_df, n_each=7)
top_neg_df = label_df[label_df["EffectSize"] < 0].copy().sort_values("EffectSize", ascending=True)
top_pos_df = label_df[label_df["EffectSize"] > 0].copy().sort_values("EffectSize", ascending=False)
fig,ax = plt.subplots(figsize=(6.7, 4.8), dpi=100)
bg_down = ewas_df[ewas_df["Status"] == "Age-associated hypomethylation"]
bg_up = ewas_df[ewas_df["Status"] == "Age-associated hypermethylation"]
bg_ns = ewas_df[ewas_df["Status"] == "Other CpGs"]
ax.scatter(bg_down["EffectSize"], bg_down["neglog10P"], s=16, alpha=0.35, c="#4F81BD", linewidth=0, label="Hypomethylation")
ax.scatter(bg_up["EffectSize"], bg_up["neglog10P"], s=16, alpha=0.35, c="#C0504D", linewidth=0, label="Hypermethylation")
ax.scatter(bg_ns["EffectSize"], bg_ns["neglog10P"], s=16, alpha=0.35, c="lightgray", linewidth=0, label="FDR Insignificant")
ax.scatter(top_neg_df["EffectSize"], top_neg_df["neglog10P"], s=30, c="#1f77b4", edgecolor="black", linewidth=0.7)
ax.scatter(top_pos_df["EffectSize"], top_pos_df["neglog10P"], s=30, c="#d62728", edgecolor="black", linewidth=0.7)
texts = []
for i, (_, row) in enumerate(label_df.iterrows()):
    txt = ax.text(row["EffectSize"], row["neglog10P"], row["AnnotationGene"], fontsize=9, alpha=0.95)
    texts.append(txt)
if len(texts) > 0:
    adjust_text(texts, ax=ax, expand_text=(1.15, 1.35), expand_points=(1.2, 1.4), force_text=(0.6, 0.9), force_points=(0.3, 0.5), arrowprops=dict(arrowstyle="-", lw=0.5, color="black", alpha=0.5))
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("Methylation β-value change per year")
ax.set_ylabel("–log10(p)")
ax.set_title("EWAS Volcano Plot of Selected CpGs in Healthy Controls")
ax.grid(alpha=0.2)
ax.legend(loc="lower right", bbox_to_anchor=(0.98, 0.05), frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

## Disease Visualizations

In [ ]:
results = dict()
residuals = dict()
control_ypred = ensemble_preds(X_test)
alzhei_ypred = ensemble_preds(X_alzhei)
schizo_ypred = ensemble_preds(X_schizo)
parkin_ypred = ensemble_preds(X_parkin)
rheuma_ypred = ensemble_preds(X_rheuma)
stroke_ypred = ensemble_preds(X_stroke)
huntin_ypred = ensemble_preds(X_huntin)
graves_ypred = ensemble_preds(X_graves)
diabet_ypred = ensemble_preds(X_diabet)
sjogre_ypred = ensemble_preds(X_sjogre)

In [ ]:
three_error_graphs(control_ypred,y_test,disease_name="Control")

In [ ]:
three_error_graphs(alzhei_ypred,alzhei_y,disease_name="Alzheimer's")

In [ ]:
three_error_graphs(schizo_ypred,schizo_y,disease_name="Schizophrenia")

In [ ]:
three_error_graphs(parkin_ypred,parkin_y,disease_name="Parkinson's")

In [ ]:
three_error_graphs(rheuma_ypred,rheuma_y,disease_name="Rheumatoid Arthritis")

In [ ]:
three_error_graphs(stroke_ypred,stroke_y,disease_name="Stroke")

In [ ]:
three_error_graphs(huntin_ypred,huntin_y,disease_name="Huntington's")

In [ ]:
three_error_graphs(graves_ypred,graves_y,disease_name="Graves'")

In [ ]:
three_error_graphs(diabet_ypred,diabet_y,disease_name="Type 2 Diabetes")

In [ ]:
three_error_graphs(sjogre_ypred,sjogre_y,disease_name="Sjögren's")

In [ ]:
cohort_info = [ ("Control", "Control", [control_ind[i] for i in rand_test]), ("Alzheimer's", "Alzheimer’s disease", alzhei_ind), ("Schizophrenia", "Schizophrenia", schizo_ind), ("Parkinson's", "Parkinson’s disease", parkin_ind), ("Rheumatoid Arthritis", "Rheumatoid arthritis", rheuma_ind), ("Stroke", "Stroke", stroke_ind), ("Huntington's", "Huntington’s disease", huntin_ind), ("Graves'", "Graves’ disease", graves_ind), ("Type 2 Diabetes", "Type 2 diabetes", diabet_ind), ("Sjögren's", "Sjögren’s syndrome", sjogre_ind) ]
rows = []
for residual_key, display_name, original_indices in cohort_info:
    if residual_key not in residuals:
        print(f"Skipping {display_name}: no residuals found for key '{residual_key}'")
        continue
    tmp = idmap.iloc[original_indices].copy()
    tmp["Cohort"] = display_name
    tmp["residual_actual_minus_predicted"] = np.asarray(residuals[residual_key])
    tmp["signed_deviation"] = -tmp["residual_actual_minus_predicted"]
    tmp["absolute_deviation"] = np.abs(tmp["residual_actual_minus_predicted"])
    rows.append(tmp)
analysis_df = pd.concat(rows, ignore_index=True)
analysis_df["Sex"] = ( analysis_df["gender"] .astype(str) .str.strip() .str.upper() .replace({ "FEMALE": "F", "MALE": "M", "NAN": "Unknown", "NONE": "Unknown", "UNKNOWN": "Unknown" }) )
analysis_df.loc[~analysis_df["Sex"].isin(["F", "M"]), "Sex"] = "Unknown"
analysis_df["age_centered"] = analysis_df["age"] - analysis_df["age"].mean()
cohort_order = [ "Control", "Alzheimer’s disease", "Schizophrenia", "Parkinson’s disease", "Rheumatoid arthritis", "Stroke", "Huntington’s disease", "Graves’ disease", "Type 2 diabetes", "Sjögren’s syndrome" ]
sex_order = ["F", "M", "Unknown"]

def mean_sd(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.2f} ({x.std():.2f})"

def median_iqr(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.median():.2f} [{x.quantile(0.25):.2f}, {x.quantile(0.75):.2f}]"

table_rows = []
for cohort in cohort_order:
    for sex in sex_order:
        g = analysis_df[(analysis_df["Cohort"] == cohort) & (analysis_df["Sex"] == sex)]
        if len(g) == 0:
            continue
        table_rows.append({ "Cohort": cohort, "Sex": sex, "N": len(g), "Age, years, mean (SD)": mean_sd(g["age"]), "Signed deviation, years, mean (SD)": mean_sd(g["signed_deviation"]), "Signed deviation, years, median [Q1, Q3]": median_iqr(g["signed_deviation"]), "Absolute deviation, years, mean (SD)": mean_sd(g["absolute_deviation"]), "Absolute deviation, years, median [Q1, Q3]": median_iqr(g["absolute_deviation"]) })
supplementary_table_4 = pd.DataFrame(table_rows)
csv_path = "/kaggle/working/supplementary_table_4_sex_stratified_residuals.csv"
supplementary_table_4.to_csv(csv_path, index=False)
supplementary_table_4

In [ ]:
import statsmodels.formula.api as smf
analysis_df["Cohort"] = pd.Categorical( analysis_df["Cohort"], categories=cohort_order, ordered=False )
analysis_df["Sex"] = pd.Categorical( analysis_df["Sex"], categories=sex_order, ordered=False )
model_df = analysis_df.dropna( subset=[ "signed_deviation", "absolute_deviation", "age_centered", "Cohort", "Sex" ] ).copy()
signed_model = smf.ols( "signed_deviation ~ " "C(Cohort, Treatment(reference='Control')) + " "age_centered + I(age_centered**2) + " "C(Sex, Treatment(reference='F'))", data=model_df ).fit(cov_type="HC3")
print(signed_model.summary())
absolute_model = smf.ols( "absolute_deviation ~ " "C(Cohort, Treatment(reference='Control')) + " "age_centered + I(age_centered**2) + " "C(Sex, Treatment(reference='F'))", data=model_df ).fit(cov_type="HC3")
print(absolute_model.summary())

def export_model_coefficients(model, path):
    ci = model.conf_int()
    coef_table = pd.DataFrame({
        "term": model.params.index,
        "estimate": model.params.values,
        "robust_SE": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "CI_lower": ci[0].values,
        "CI_upper": ci[1].values
    })
    coef_table.to_csv(path, index=False)
    return coef_table

signed_coef_table = export_model_coefficients( signed_model, "/kaggle/working/sex_age_adjusted_signed_deviation_coefficients.csv" )
absolute_coef_table = export_model_coefficients( absolute_model, "/kaggle/working/sex_age_adjusted_absolute_deviation_coefficients.csv" )
signed_coef_table

In [ ]:
vresults = sorted(results.items(), key=lambda v: v[1])
dis_names = [k for k,v in vresults]
dis_mae = [v for k,v in vresults]
vresults

## Disease Comparison

In [ ]:
def confidence_ints(true_y,pred_y):
    maes = []
    for _ in range(10000):
        y_resample, pred_resample = resample(true_y, pred_y)
        maes.append(mean_absolute_error(y_resample, pred_resample))
    ci_lower,ci_upper = np.percentile(maes, [2.5,97.5])
    return round(ci_lower,3),round(ci_upper,3)

In [ ]:
confident_results = {k:[results[k]] for k in results}
confident_results["Control"] += list(confidence_ints(y_test,control_ypred))
confident_results["Alzheimer's"] += list(confidence_ints(alzhei_y,alzhei_ypred))
confident_results["Schizophrenia"] += list(confidence_ints(schizo_y,schizo_ypred))
confident_results["Parkinson's"] += list(confidence_ints(parkin_y,parkin_ypred))
confident_results["Rheumatoid Arthritis"] += list(confidence_ints(rheuma_y,rheuma_ypred))
confident_results["Stroke"] += list(confidence_ints(stroke_y,stroke_ypred))
confident_results["Huntington's"] += list(confidence_ints(huntin_y,huntin_ypred))
confident_results["Graves'"] += list(confidence_ints(graves_y,graves_ypred))
confident_results["Type 2 Diabetes"] += list(confidence_ints(diabet_y,diabet_ypred))
confident_results["Sjögren's"] += list(confidence_ints(sjogre_y,sjogre_ypred))
vcresults = dict(sorted(confident_results.items(), key=lambda v: v[1]))
vcresults

In [ ]:
mae_means = []
error = []
for vs in vcresults.values():
    mae_means.append(vs[0])
    error.append([vs[0]-vs[1],vs[2]-vs[0]])

In [ ]:
plt.figure(figsize=(6.1, 4.8))
plt.bar(vcresults.keys(), mae_means, yerr=np.array(error).T, capsize=8, color='#40B0FF', edgecolor='black')
plt.xlabel("Disease")
plt.ylabel("Mean Absolute Error (MAE)")
plt.title("Disease MAE Comparison with 95% Confidence Intervals")
plt.xticks(rotation=90, ha='center')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
vresiduals = dict()
for dis in vcresults.keys():
    vresiduals[dis] = residuals[dis]
residual_data = []
for dd, rr in vresiduals.items():
    for r in rr:
        residual_data.append({'Disease': dd, 'Residual': r})
res_df = pd.DataFrame(residual_data)
res_df['Residual'] = res_df['Residual'].clip(-15, 20)
res_sample = res_df.groupby("Disease").apply(lambda x: x.sample(n=min(len(x), 100), random_state=42)).reset_index(drop=True)

In [ ]:
plt.figure(figsize=(8, 6))
sns.violinplot(data=res_sample, x='Disease', y='Residual', inner="box", color='#40B0FF', density_norm="width", cut=1, order=list(vcresults.keys()), inner_kws=dict(box_width=7, whis_width=1))
plt.title("Residual Distributions by Disease")
plt.xlabel("Disease")
plt.ylabel("Residual (Actual - Predicted)")
plt.xticks(rotation=90, ha='center')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## Disease Biomarkers

In [ ]:
get_cpg_info('cg04630292')

In [ ]:
def top_biomarkers_cathyp(X_disease, disease, n_components=10):
    ideal = {"Graves'":3,"Alzheimer's":5,"Rheumatoid Arthritis":5,"Stroke":5,"Huntington's":5,"Parkinson's":7,"Sjögren's":10,"Type 2 Diabetes":10,"Schizophrenia":10}
    pacmap_neighbors = ideal[disease]
    T_dis = np.vstack([X_train, X_disease]).astype(np.float32)
    Ty_dis = np.concatenate([np.zeros(len(X_train)), np.ones(len(X_disease))]).astype(int)
    S_dis = np.vstack([X_test, X_disease]).astype(np.float32)
    Sy_dis = np.concatenate([np.zeros(len(X_test)), np.ones(len(X_disease))]).astype(int)
    pac = pacmap.PaCMAP(n_components=n_components,n_neighbors=pacmap_neighbors,MN_ratio=0.5,FP_ratio=2.0,random_state=42)
    T_emb = pac.fit_transform(T_dis, init="pca").astype(np.float32)
    S_emb = pac.transform(S_dis, basis=T_dis).astype(np.float32)
    cat_class = CatBoostClassifier(iterations=1000, min_data_in_leaf=1, learning_rate=0.03, max_leaves=25, l2_leaf_reg=0.1, grow_policy="Lossguide", logging_level="Silent", task_type="GPU")
    cat_class.fit(T_emb,Ty_dis)
    cat_pred = cat_class.predict(S_emb).astype(int)
    cat_prob = cat_class.predict_proba(S_emb)[:,1]
    plt.figure(figsize=(1.5, 1.25), dpi=100)
    plt.title(disease, size=8)
    sns.heatmap(confusion_matrix(Sy_dis, cat_pred), annot=True, fmt="g", cmap="Blues", square=True, vmin=0, vmax=2000)
    plt.show()
    acc = accuracy_score(Sy_dis, cat_pred)
    f1  = f1_score(Sy_dis, cat_pred)
    auc = roc_auc_score(Sy_dis, cat_prob)
    print(f"{disease} was classified with {acc*100:.2f}% accuracy and an f1-score of {f1*100:.2f}%.")
    print(f"The AUROC was {auc:.3f}.")
    feature_names = [f"PaCMAP_{i}" for i in range(n_components)]
    rng = np.random.RandomState(42)
    bg_n = min(200, T_emb.shape[0])
    bg_idx = rng.choice(T_emb.shape[0], bg_n, replace=False)
    background = T_emb[bg_idx]
    explain_n = min(500, S_emb.shape[0])
    ex_idx = rng.choice(S_emb.shape[0], explain_n, replace=False)
    S_sub = S_emb[ex_idx]
    explainer = shap.Explainer(cat_class.predict_proba, background, algorithm="permutation")
    shap_exp = explainer(S_sub)
    shap_vals_pos = shap_exp.values
    if shap_vals_pos.ndim == 3:
        shap_vals_pos = shap_vals_pos[:, :, 1]
    shap.summary_plot(shap_vals_pos, S_sub,
                      feature_names=feature_names,
                      max_display=min(10, n_components))
    try:
        mean_abs = np.mean(np.abs(shap_vals_pos), axis=0)
        dim_order = np.argsort(-mean_abs)[:min(5, n_components)]
        Xc = T_dis[:len(X_train)].astype(np.float32)
        Zc = T_emb[:len(X_train)].astype(np.float32)
        Xc = (Xc - Xc.mean(axis=0)) / (Xc.std(axis=0) + 1e-6)
        Zc = (Zc - Zc.mean(axis=0)) / (Zc.std(axis=0) + 1e-6)
        corr = (Xc.T @ Zc) / (Xc.shape[0] - 1)
        chosen = []
        chosen_set = set()
        print("\nTop 5 gene-annotated proxy CpGs across most important PaCMAP dimensions:")
        for d in dim_order:
            idx_sorted = np.argsort(-np.abs(corr[:, d]))
            for j in idx_sorted:
                cg = cpg_data[j]
                if cg in chosen_set:
                    continue
                gene = ""
                try:
                    raw = get_cpg_info(cg)["UCSC_RefGene_Name"].iloc[0]
                    raw = "" if raw is None else str(raw)
                    if raw and raw.lower() != "nan":
                        gene = raw.split(";")[0].strip()
                except:
                    gene = ""
                if not gene:
                    continue
                chosen.append((cg, gene, d, float(corr[j, d]), float(mean_abs[d])))
                chosen_set.add(cg)
                if len(chosen) >= 5:
                    break
            if len(chosen) >= 5:
                break
        for cg, gene, d, r, ms in chosen:
            print(f"  {cg}  ->  {gene}   (PaCMAP_{d}, corr={r:+.3f}, mean|SHAP|={ms:.4f})")
        if len(chosen) < 5:
            print(f"\n[Note] Only found {len(chosen)} CpGs with UCSC gene annotations using get_cpg_info().")
    except Exception as e:
        print(f"\n[Note] Skipped proxy CpG mapping due to: {e}")

In [ ]:
top_biomarkers_cathyp(X_alzhei,"Alzheimer's")

In [ ]:
top_biomarkers_cathyp(X_schizo,"Schizophrenia")

In [ ]:
top_biomarkers_cathyp(X_parkin,"Parkinson's")

In [ ]:
top_biomarkers_cathyp(X_rheuma,"Rheumatoid Arthritis")

In [ ]:
top_biomarkers_cathyp(X_stroke,"Stroke")

In [ ]:
top_biomarkers_cathyp(X_huntin,"Huntington's")

In [ ]:
top_biomarkers_cathyp(X_graves,"Graves'")

In [ ]:
top_biomarkers_cathyp(X_diabet,"Type 2 Diabetes")

In [ ]:
top_biomarkers_cathyp(X_sjogre,"Sjögren's")

In [ ]:
disease_info = [
    ("Alzheimer’s disease", alzhei_ind),
    ("Schizophrenia", schizo_ind),
    ("Parkinson’s disease", parkin_ind),
    ("Rheumatoid arthritis", rheuma_ind),
    ("Stroke", stroke_ind),
    ("Huntington’s disease", huntin_ind),
    ("Graves’ disease", graves_ind),
    ("Type 2 diabetes", diabet_ind),
    ("Sjögren’s syndrome", sjogre_ind)
]

def clean_sex(x):
    x = str(x).strip().upper()
    if x in ["F", "FEMALE"]:
        return "Female"
    if x in ["M", "MALE"]:
        return "Male"
    return "Unknown"

baseline_rows = []

for disease_name, disease_indices in disease_info:
    all_indices = list(control_ind) + list(disease_indices)

    demo_df = idmap.iloc[all_indices][["age", "gender"]].copy()
    demo_df["Sex"] = demo_df["gender"].apply(clean_sex)
    demo_df = demo_df[["age", "Sex"]]

    y_binary = np.array([0] * len(control_ind) + [1] * len(disease_indices))

    numeric_features = ["age"]
    categorical_features = ["Sex"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    y_pred = cross_val_predict(clf, demo_df, y_binary, cv=cv, method="predict")
    y_prob = cross_val_predict(clf, demo_df, y_binary, cv=cv, method="predict_proba")[:, 1]

    baseline_rows.append({
        "Disease": disease_name,
        "Control N": len(control_ind),
        "Disease N": len(disease_indices),
        "Age and sex accuracy": accuracy_score(y_binary, y_pred),
        "Age and sex balanced accuracy": balanced_accuracy_score(y_binary, y_pred),
        "Age and sex F1": f1_score(y_binary, y_pred, pos_label=1, zero_division=0),
        "Age and sex precision": precision_score(y_binary, y_pred, pos_label=1, zero_division=0),
        "Age and sex recall": recall_score(y_binary, y_pred, pos_label=1, zero_division=0),
        "Age and sex AUROC": roc_auc_score(y_binary, y_prob)
    })

age_sex_baseline_table = pd.DataFrame(baseline_rows)

metric_cols = [
    "Age and sex accuracy",
    "Age and sex balanced accuracy",
    "Age and sex F1",
    "Age and sex precision",
    "Age and sex recall",
    "Age and sex AUROC"
]

age_sex_baseline_table[metric_cols] = age_sex_baseline_table[metric_cols].round(3)

age_sex_baseline_table.to_csv(
    "/kaggle/working/age_sex_only_disease_classifier_baseline.csv",
    index=False
)

age_sex_baseline_table